# Debug Optimizer Notebook

Use this notebook to debug bullet extraction, section detection, and replacement issues.

## Setup
1. Paste your LaTeX resume into `debug_data/sample_resume.tex`
2. Run cells to diagnose issues

In [16]:
# Setup
%load_ext autoreload
%autoreload 2

import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path("../backend").resolve()))
logging.basicConfig(level=logging.INFO)

from tailor_tom.optimizer import (
    _extract_bullet_constraints,
    _apply_replacement_to_latex,
    _validate_replacement,
    _format_bullets_for_llm,
    _strip_latex_commands,
)
from tailor_tom.layout_analyzer import extract_items_from_latex, extract_line_metrics
from tailor_tom.latex_compiler import compile_latex

print("Setup complete!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Setup complete!


In [17]:
# Load LaTeX
latex_path = Path("debug_data/sample_resume.tex")
latex = latex_path.read_text(encoding="utf-8") if latex_path.exists() else None

if latex:
    print(f"Loaded: {len(latex)} chars, {latex.count(chr(10))} lines")
else:
    print(f"ERROR: Create {latex_path} with your LaTeX resume")

Loaded: 15922 chars, 278 lines


In [ ]:
# Job description placeholder
job_description = """
Software Engineering Intern - Summer 2026
New York City
Squarespace is looking for passionate and eager students to join our Software Engineering Internship Program this summer in New York City. We have a variety of evolving teams at Squarespace who are pushing out the next generation of web platform technologies. Over the course of our 12 week paid program, you will contribute, grow your technical skills, and explore all that Squarespace and NYC have to offer!

Our Intern Role
Frontend: Our Frontend teams build one of the most advanced interfaces on the web with an intricate eye for design. For Frontend, you will need experience in JavaScript.

How to Apply
Please review team descriptions and note the team language requirement.
We'll review your application, and if your experience is a match, we'll send you a take-home assessment before moving on to interviews.
Qualifications for All Roles
Currently enrolled student, graduating in December 2026 or Spring 2027.
Pursuing a BS or MS in Computer Science, Computer Engineering, or a related technical discipline.
A foundation in computer science with competencies in data structures, algorithms, and software design practices.
Compensation: $54.00 USD hourly rate
About Squarespace
Squarespace is a design-driven platform helping entrepreneurs build brands and businesses online. We empower millions of customers in more than 200 countries and territories with all the tools they need to create an online presence, build an audience, monetize, and scale their business. Our suite of products range from websites, domains, ecommerce, and marketing tools, as well as tools for scheduling with Acuity and creating and managing social media presence with Bio Sites and Unfold. Our team of more than 1,700 is headquartered in New York City, with offices in Dublin, Ireland, and Aveiro, Portugal. For more information about our company, visit https://www.squarespace.com/about/careers.

Our Commitment
Today, more than a million people around the globe use Squarespace to share different perspectives and experiences with the world. Not only do we embrace and celebrate the diversity of our customers, but we also work toward the same in our employees. At Squarespace, we are committed to equal employment opportunity regardless of race, color, ethnicity, ancestry, religion, national origin, gender, sex, gender identity or expression, sexual orientation, age, citizenship, marital or parental status, disability, veteran status, or other class protected by applicable law. We are proud to be an equal opportunity workplace.

#LI-Hybrid

Thank you in advance for providing the following details about your work history from your resume! This helps us ensure that your candidate information is accurate and consistent during the hiring process.

 

Squarespace will never solicit your personal banking information or ask you to transfer money in connection with a job offer or interview. We also will not reach out to you via phone or SMS without your permission or knowledge.
"""
print(f"Job description: {len(job_description)} chars")

## Debug: Bullet Extraction

Compare LaTeX items vs PDF bullets to find mismatches.

In [ ]:
# Extract from LaTeX source
if latex:
    latex_items = extract_items_from_latex(latex)
    print(f"LaTeX items: {len(latex_items)}\n")
    for i, item in enumerate(latex_items[:10]):  # First 10
        print(f"{i+1}. {item.get('text', '')[:70]}...")

In [ ]:
# Compile and extract from PDF
pdf_bytes = None
if latex:
    result = compile_latex(latex)
    if result.success:
        pdf_bytes = result.pdf_bytes
        Path("debug_data/compiled_resume.pdf").write_bytes(pdf_bytes)
        
        pdf_metrics = extract_line_metrics(pdf_bytes, latex=latex)
        pdf_bullets = pdf_metrics.get("bullets", [])
        print(f"PDF bullets: {len(pdf_bullets)}")
        print(f"Match: {'YES' if len(latex_items) == len(pdf_bullets) else 'NO - MISMATCH!'}")
    else:
        print(f"Compilation failed: {result.error_message}")

## Debug: Constraints & Section Detection

Check if bullets are correctly assigned to sections.

In [ ]:
# Build constraints and check sections
constraints = []
if pdf_bytes and latex:
    constraints = _extract_bullet_constraints(pdf_bytes, latex)
    
    # Group by section
    from collections import Counter
    sections = Counter(c.section for c in constraints)
    print("Section breakdown:")
    for section, count in sections.most_common():
        editable = "(editable)" if section not in ("Education", "Skills") else "(PROTECTED)"
        print(f"  {section}: {count} bullets {editable}")
    
    print(f"\nEditable: {sum(1 for c in constraints if c.section not in ('Education', 'Skills'))}")
    print(f"Protected: {sum(1 for c in constraints if c.section in ('Education', 'Skills'))}")

In [ ]:
# Show all constraints with sections
for c in constraints:
    status = "SKIP" if c.section in ("Education", "Skills") else "EDIT"
    print(f"[{status}] B{c.bullet_id} [{c.section}] {c.word_count}w: {c.original_text[:50]}...")

In [ ]:
# Test LaTeX formatting preservation
if constraints and latex:
    TEST_ID = 3  # Change to test different bullets
    c = next((c for c in constraints if c.bullet_id == TEST_ID), None)
    
    if c and c.latex_snippet:
        print(f"=== LaTeX Formatting Preservation Test for Bullet {TEST_ID} ===\n")
        
        # Show original
        print(f"Original LaTeX:\n{c.latex_snippet}\n")
        
        # Strip and show plain text
        plain = _strip_latex_commands(c.latex_snippet)
        print(f"Stripped (for counting): {plain}\n")
        print(f"Word count: {len(plain.split())}, Char count: {len(plain)}")
        
        # Create a test replacement with formatting
        test_latex = c.latex_snippet.replace("first", "initial").replace("and", "plus")
        print(f"\nTest replacement LaTeX:\n{test_latex}\n")
        
        # Validate
        is_valid, reason = _validate_replacement(c, test_latex)
        print(f"Validation: {'PASS' if is_valid else 'FAIL'} - {reason or 'OK'}")
        
        # Apply
        new_doc, success = _apply_replacement_to_latex(latex, c, test_latex)
        print(f"Apply: {'SUCCESS' if success else 'FAILED'}")

## Debug: Replacement Testing

Test if replacements can be applied to specific bullets.

In [ ]:
# Test replacement on a specific bullet
TEST_ID = 3  # Change to test different bullets

if constraints and latex:
    c = next((c for c in constraints if c.bullet_id == TEST_ID), None)
    if c:
        print(f"Bullet {TEST_ID} [{c.section}]")
        print(f"Original ({c.word_count}w): {c.original_text[:80]}...")
        print(f"\nLaTeX snippet:")
        print(c.latex_snippet[:200])
        
        # Test a dummy replacement
        test_text = "Test replacement with similar word count to original bullet text here"
        new_latex, success = _apply_replacement_to_latex(latex, c, test_text)
        print(f"\nReplacement test: {'SUCCESS' if success else 'FAILED'}")

## Debug: LLM Input Preview

See what gets sent to the LLM.

In [ ]:
# Preview LLM input
if constraints:
    editable = [c for c in constraints if c.section not in ("Education", "Skills")]
    print(f"Sending {len(editable)} bullets to LLM\n")
    
    formatted = _format_bullets_for_llm(editable, max_word_delta=3)
    print(formatted[:1500])

## Debug: Manual Inspection

In [ ]:
# Search for text in LaTeX
SEARCH = "Conducted"  # Change this

if latex:
    for i, line in enumerate(latex.split("\n")):
        if SEARCH.lower() in line.lower():
            print(f"L{i+1}: {line[:100]}")

In [ ]:
# Count item commands
import re
if latex:
    patterns = [r'\\resumeItem\{', r'\\item\s', r'\\item\{']
    for p in patterns:
        print(f"{p}: {len(re.findall(p, latex))}")

In [ ]:
# Export constraints to CSV
import csv
if constraints:
    with open("debug_data/constraints.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "section", "words", "chars", "lines", "text"])
        for c in constraints:
            w.writerow([c.bullet_id, c.section, c.word_count, c.char_count, c.line_count, c.original_text[:100]])
    print("Exported to debug_data/constraints.csv")